In [1]:
import xarray as xr

In [2]:
data = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/OM4_5daily_v0.2.1.zarr"
)

# data = xr.open_zarr("/vast/sd5313/data/m2lines/3D_ocean_data/test_CMIP6_GFDL-CM4.piControl.r1i1p1f1.zarr")

In [3]:
data

<xarray.Dataset>
Dimensions:         (y: 180, x: 360, lev: 19, time: 4745, y_b: 181, x_b: 361)
Coordinates:
    areacello       (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    dz              (lev) int64 dask.array<chunksize=(19,), meta=np.ndarray>
    lat             (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    lat_b           (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
  * lev             (lev) float64 2.5 10.0 22.5 40.0 ... 4e+03 5e+03 6e+03
    lon             (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    lon_b           (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
    ocean_fraction  (lev, y, x) float64 dask.array<chunksize=(19, 180, 360), meta=np.ndarray>
  * time            (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
    wetmask         (lev, y, x) bool dask.array<chunksize=(10, 90, 360), meta=np.ndarray>
  * x               (x) float64 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * y               (y) float64 -89.24 -88.25 -87.25 ... 87.25 88.25 89.24
Dimensions without coordinates: y_b, x_b
Data variables:
    hfds            (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so              (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    tauuo           (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauvo           (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao          (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    uo              (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    vo              (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    zos             (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
Attributes:
    m2lines/ocean-emulators_git_hash:  https://github.com/m2lines/ocean_emula...
    regrid_method:                     conservative

### Convert and save 3D data to /vast

Make sure you are using atleast 10 cores!

In [4]:
from dask.diagnostics import ProgressBar

In [5]:
ds = data

In [33]:
ds["lev"]

<xarray.DataArray 'lev' (lev: 19)>
array([2.50e+00, 1.00e+01, 2.25e+01, 4.00e+01, 6.50e+01, 1.05e+02, 1.65e+02,
       2.50e+02, 3.75e+02, 5.50e+02, 7.75e+02, 1.05e+03, 1.40e+03, 1.85e+03,
       2.40e+03, 3.10e+03, 4.00e+03, 5.00e+03, 6.00e+03])
Coordinates:
    dz       (lev) int64 dask.array<chunksize=(19,), meta=np.ndarray>
  * lev      (lev) float64 2.5 10.0 22.5 40.0 65.0 ... 3.1e+03 4e+03 5e+03 6e+03

In [40]:
assert [str(lev).replace(".", "_") for lev in ds["lev"].values] == ['2_5', '10_0', '22_5', '40_0', '65_0', '105_0', '165_0', '250_0', '375_0', '550_0', '775_0', '1050_0', '1400_0', '1850_0', '2400_0', '3100_0', '4000_0', '5000_0', '6000_0']

In [6]:
for lev in ds["lev"].values:
    lev_str = str(lev).replace(".", "_")

    # Create new variables for each original variable with the lev dimension
    ds[f"vo_lev_{lev_str}"] = ds["vo"].sel(lev=lev)
    ds[f"thetao_lev_{lev_str}"] = ds["thetao"].sel(lev=lev)
    ds[f"uo_lev_{lev_str}"] = ds["uo"].sel(lev=lev)
    ds[f"so_lev_{lev_str}"] = ds["so"].sel(lev=lev)

ds = ds.drop_vars(["vo", "thetao", "uo", "so"])
ds

<xarray.Dataset>
Dimensions:            (y: 180, x: 360, lev: 19, time: 4745, y_b: 181, x_b: 361)
Coordinates:
    areacello          (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    dz                 (lev) int64 dask.array<chunksize=(19,), meta=np.ndarray>
    lat                (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    lat_b              (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
  * lev                (lev) float64 2.5 10.0 22.5 40.0 ... 4e+03 5e+03 6e+03
    lon                (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    lon_b              (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
    ocean_fraction     (lev, y, x) float64 dask.array<chunksize=(19, 180, 360), meta=np.ndarray>
  * time               (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
    wetmask            (lev, y, x) bool dask.array<chunksize=(10, 90, 360), meta=np.ndarray>
  * x                  (x) float64 0.5 1.5 2.5 3.5 ... 356.5 357.5 358.5 359.5
  * y                  (y) float64 -89.24 -88.25 -87.25 ... 87.25 88.25 89.24
Dimensions without coordinates: y_b, x_b
Data variables: (12/80)
    hfds               (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauuo              (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauvo              (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    zos                (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_2_5         (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao_lev_2_5     (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    ...                 ...
    uo_lev_5000_0      (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_5000_0      (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_6000_0      (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao_lev_6000_0  (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    uo_lev_6000_0      (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_6000_0      (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
Attributes:
    m2lines/ocean-emulators_git_hash:  https://github.com/m2lines/ocean_emula...
    regrid_method:                     conservative

In [7]:
with ProgressBar():
    ds_mean = ds.mean().compute()

[########################################] | 100% Completed | 519.34 s


In [8]:
ds_mean.to_zarr("/pscratch/sd/s/suryad/data/3D_data_OM4_5daily_v0.2.1_means")

In [9]:
with ProgressBar():
    ds_std = ds.std().compute()

[########################################] | 100% Completed | 11m 11s


In [10]:
ds_std.to_zarr("/pscratch/sd/s/suryad/data/3D_data_OM4_5daily_v0.2.1_stds")

In [11]:
with ProgressBar():
    ds.to_zarr("/pscratch/sd/s/suryad/data/3D_data_OM4_5daily_v0.2.1")

[########################################] | 100% Completed | 483.16 s


In [32]:
ds

<xarray.Dataset>
Dimensions:            (y: 180, x: 360, lev: 19, time: 4745, y_b: 181, x_b: 361)
Coordinates:
    areacello          (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    dz                 (lev) int64 dask.array<chunksize=(19,), meta=np.ndarray>
    lat                (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    lat_b              (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
  * lev                (lev) float64 2.5 10.0 22.5 40.0 ... 4e+03 5e+03 6e+03
    lon                (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    lon_b              (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
    ocean_fraction     (lev, y, x) float64 dask.array<chunksize=(19, 180, 360), meta=np.ndarray>
  * time               (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
    wetmask            (lev, y, x) bool dask.array<chunksize=(10, 90, 360), meta=np.ndarray>
  * x                  (x) float64 0.5 1.5 2.5 3.5 ... 356.5 357.5 358.5 359.5
  * y                  (y) float64 -89.24 -88.25 -87.25 ... 87.25 88.25 89.24
Dimensions without coordinates: y_b, x_b
Data variables: (12/80)
    hfds               (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauuo              (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauvo              (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    zos                (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_2_5         (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao_lev_2_5     (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    ...                 ...
    uo_lev_5000_0      (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_5000_0      (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_6000_0      (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao_lev_6000_0  (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    uo_lev_6000_0      (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_6000_0      (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
Attributes:
    m2lines/ocean-emulators_git_hash:  https://github.com/m2lines/ocean_emula...
    regrid_method:                     conservative

#### Wet mask

In [12]:
import xarray as xr

In [13]:
data = xr.open_zarr("/pscratch/sd/s/suryad/data/OM4_5daily_v0.2.1.zarr")
levels = 19
# data = xr.open_zarr(
#     "/vast/sd5313/data/m2lines/3D_ocean_data/test_CMIP6_GFDL-CM4.piControl.r1i1p1f1.zarr"
# )
data

<xarray.Dataset>
Dimensions:         (y: 180, x: 360, lev: 19, time: 4745, y_b: 181, x_b: 361)
Coordinates:
    areacello       (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    dz              (lev) int64 dask.array<chunksize=(19,), meta=np.ndarray>
    lat             (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    lat_b           (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
  * lev             (lev) float64 2.5 10.0 22.5 40.0 ... 4e+03 5e+03 6e+03
    lon             (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    lon_b           (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
    ocean_fraction  (lev, y, x) float64 dask.array<chunksize=(19, 180, 360), meta=np.ndarray>
  * time            (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
    wetmask         (lev, y, x) bool dask.array<chunksize=(10, 90, 360), meta=np.ndarray>
  * x               (x) float64 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * y               (y) float64 -89.24 -88.25 -87.25 ... 87.25 88.25 89.24
Dimensions without coordinates: y_b, x_b
Data variables:
    hfds            (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so              (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    tauuo           (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauvo           (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao          (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    uo              (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    vo              (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    zos             (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
Attributes:
    m2lines/ocean-emulators_git_hash:  https://github.com/m2lines/ocean_emula...
    regrid_method:                     conservative

In [14]:
data = data.drop(["tauuo", "tauvo", "hfds"])
# data = data.drop(["tauuo", "tauvo", "hft"])
data

<xarray.Dataset>
Dimensions:         (y: 180, x: 360, lev: 19, y_b: 181, x_b: 361, time: 4745)
Coordinates:
    areacello       (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    dz              (lev) int64 dask.array<chunksize=(19,), meta=np.ndarray>
    lat             (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    lat_b           (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
  * lev             (lev) float64 2.5 10.0 22.5 40.0 ... 4e+03 5e+03 6e+03
    lon             (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    lon_b           (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
    ocean_fraction  (lev, y, x) float64 dask.array<chunksize=(19, 180, 360), meta=np.ndarray>
  * time            (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
    wetmask         (lev, y, x) bool dask.array<chunksize=(10, 90, 360), meta=np.ndarray>
  * x               (x) float64 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * y               (y) float64 -89.24 -88.25 -87.25 ... 87.25 88.25 89.24
Dimensions without coordinates: y_b, x_b
Data variables:
    so              (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    thetao          (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    uo              (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    vo              (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    zos             (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
Attributes:
    m2lines/ocean-emulators_git_hash:  https://github.com/m2lines/ocean_emula...
    regrid_method:                     conservative

In [15]:
import numpy as np
import torch


def get_wet_mask(inputs, device="cpu"):
    wet = xr.zeros_like(inputs[0][0])
    # inputs[0][0,12,12] = np.nan
    for data in inputs:
        wet += np.isnan(data[0])

    wet_nan = xr.where(wet != 0, np.nan, 1).to_numpy()
    wet = np.isnan(xr.where(wet == 0, np.nan, 0))
    wet = np.nan_to_num(wet.to_numpy())
    wet = torch.from_numpy(wet).type(torch.float32).to(device=device)
    return wet, wet_nan

In [16]:
wet_stacked = []
for i in range(levels):
    inputs = []
    inputs.append(data["uo"][:, i])
    inputs.append(data["vo"][:, i])
    inputs.append(data["thetao"][:, i])
    inputs.append(data["so"][:, i])
    if i == 0:
        inputs.append(data["zos"])

    inputs = tuple(inputs)
    wet, _ = get_wet_mask(inputs)
    wet_stacked.append(wet)

In [17]:
wet_3D = torch.stack(wet_stacked)

In [18]:
out_list = [
    k + str(j)
    for k in ["uo_lev_", "vo_lev_", "thetao_lev_", "so_lev_"]
    for j in range(levels)
] + ["zos"]
out_list

['uo_lev_0',
 'uo_lev_1',
 'uo_lev_2',
 'uo_lev_3',
 'uo_lev_4',
 'uo_lev_5',
 'uo_lev_6',
 'uo_lev_7',
 'uo_lev_8',
 'uo_lev_9',
 'uo_lev_10',
 'uo_lev_11',
 'uo_lev_12',
 'uo_lev_13',
 'uo_lev_14',
 'uo_lev_15',
 'uo_lev_16',
 'uo_lev_17',
 'uo_lev_18',
 'vo_lev_0',
 'vo_lev_1',
 'vo_lev_2',
 'vo_lev_3',
 'vo_lev_4',
 'vo_lev_5',
 'vo_lev_6',
 'vo_lev_7',
 'vo_lev_8',
 'vo_lev_9',
 'vo_lev_10',
 'vo_lev_11',
 'vo_lev_12',
 'vo_lev_13',
 'vo_lev_14',
 'vo_lev_15',
 'vo_lev_16',
 'vo_lev_17',
 'vo_lev_18',
 'thetao_lev_0',
 'thetao_lev_1',
 'thetao_lev_2',
 'thetao_lev_3',
 'thetao_lev_4',
 'thetao_lev_5',
 'thetao_lev_6',
 'thetao_lev_7',
 'thetao_lev_8',
 'thetao_lev_9',
 'thetao_lev_10',
 'thetao_lev_11',
 'thetao_lev_12',
 'thetao_lev_13',
 'thetao_lev_14',
 'thetao_lev_15',
 'thetao_lev_16',
 'thetao_lev_17',
 'thetao_lev_18',
 'so_lev_0',
 'so_lev_1',
 'so_lev_2',
 'so_lev_3',
 'so_lev_4',
 'so_lev_5',
 'so_lev_6',
 'so_lev_7',
 'so_lev_8',
 'so_lev_9',
 'so_lev_10',
 'so_lev_11'

In [19]:
# print(out_list[:38] + out_list[39:])

In [20]:
final_wet = []
for var in out_list:
    try:
        level = int(var.split("lev_")[-1])
    except:
        level = 0
    final_wet.append(wet_3D[level])

In [22]:
wet = torch.stack(final_wet)
wet.shape

torch.Size([77, 180, 360])

In [23]:
assert wet.shape[0] == (levels * 4 + 1)

In [24]:
# wet = torch.cat([wet[:38], wet[39:]], axis=0)
# wet.shape

In [25]:
torch.save(wet, "/pscratch/sd/s/suryad/data/3D_wet_OM4_5daily_v0.2.1.pt".format(levels))

### Convert and save surface data

Make sure you are using atleast 8 cores!

In [26]:
import sys

sys.path.append("../src/")

In [27]:
from utils.data_utils import get_wet_mask

In [28]:
inputs = []
inputs.append(data["uo"][:, 0])
inputs.append(data["vo"][:, 0])
inputs.append(data["thetao"][:, 0])
inputs.append(data["so"][:, 0])
inputs.append(data["zos"])
inputs = tuple(inputs)
wet, _ = get_wet_mask(inputs)
print("Wet resolution:", wet.shape)

Wet resolution: torch.Size([180, 360])


In [ ]:
# print("Calculating mask tensors")
# wet, wet_nan = get_wet_mask(inputs, "cpu")
# # wet_bool = np.array(wet.cpu()).astype(bool)
# # wet_lap = compute_laplacian_wet(wet_nan, 4)  # hardcoded
# # wet_lap = xr.where(wet_lap == 0, 1, np.nan)
# # wet_lap = np.nan_to_num(wet_lap)
# print("Wet resolution:", wet.shape)

Calculating mask tensors
Wet resolution: torch.Size([180, 360])


In [29]:
import torch

In [30]:
torch.save(wet, "/pscratch/sd/s/suryad/data/surface_wet_OM4_5daily_v0.2.1.pt")

In [ ]:
from dask.diagnostics import ProgressBar

In [ ]:
with ProgressBar():
    data_mean = data.mean().compute()

[########################################] | 100% Completed | 259.26 s


In [ ]:
data_mean.to_zarr("/vast/sd5313/data/m2lines/3D_ocean_data/surface_data_means")

In [ ]:
with ProgressBar():
    data_std = data.std().compute()

[########################################] | 100% Completed | 244.16 s


In [ ]:
data_std.to_zarr("/vast/sd5313/data/m2lines/3D_ocean_data/surface_data_stds")

In [ ]:
data.to_zarr("/vast/sd5313/data/m2lines/3D_ocean_data/surface_data")

### Test

In [44]:
data = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/OM4_5daily_v0.2.1.zarr"
)
data.astype("float32")


<xarray.Dataset>
Dimensions:         (time: 4745, y: 180, x: 360, lev: 19, y_b: 181, x_b: 361)
Coordinates:
    areacello       (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    dz              (lev) int64 dask.array<chunksize=(19,), meta=np.ndarray>
    lat             (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    lat_b           (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
  * lev             (lev) float64 2.5 10.0 22.5 40.0 ... 4e+03 5e+03 6e+03
    lon             (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    lon_b           (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
    ocean_fraction  (lev, y, x) float64 dask.array<chunksize=(19, 180, 360), meta=np.ndarray>
  * time            (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
    wetmask         (lev, y, x) bool dask.array<chunksize=(10, 90, 360), meta=np.ndarray>
  * x               (x) float64 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * y               (y) float64 -89.24 -88.25 -87.25 ... 87.25 88.25 89.24
Dimensions without coordinates: y_b, x_b
Data variables:
    hfds            (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so              (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    tauuo           (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauvo           (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao          (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    uo              (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    vo              (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    zos             (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
Attributes:
    m2lines/ocean-emulators_git_hash:  https://github.com/m2lines/ocean_emula...
    regrid_method:                     conservative

In [43]:
data = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/OM4_Horizontal_Regrid_Old.zarr"
)
data

<xarray.Dataset>
Dimensions:  (time: 4745, y: 180, x: 360, lev: 19)
Coordinates:
  * time     (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
  * x        (x) float64 0.5 1.5 2.5 3.5 4.5 ... 355.5 356.5 357.5 358.5 359.5
  * y        (y) float64 -89.5 -88.5 -87.5 -86.5 -85.5 ... 86.5 87.5 88.5 89.5
Dimensions without coordinates: lev
Data variables:
    hfds     (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    tauuo    (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauvo    (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao   (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    uo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    vo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    zos      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>